# 2.2. Подбор и изучение источников данных

Тема: «Исследование открытых данных Федерального казначейства для обнаружения структурных и временных паттернов бюджетных потоков».

На этом этапе выбираются источники разных форматов, проверяется их структура и определяются признаки для дальнейшего анализа.

Используются:
1. табличные данные Федерального казначейства
2. табличные данные Росстата как факторы влияния
3. API "budget.gov.ru" с JSON
4. XML Банка России
5. неструктурированные материалы Росстата


## Подготовка окружения

Файл "treasury.zip" должен лежать в папке "data" рядом с ноутбуком.

Архив Росстата скачан в формате RAR. Он был распакован в папку "data/rosstat".

XML Банка России сохранен в папку "data" как "cbr_rates.xml".

Доклад Росстата сохранен в папку "data" как "rosstat_report.pdf".

Итоговая структура:

```text
data/
├── treasury.zip
├── rosstat.rar
├── rosstat/
│   └── ... файлы Росстата ...
├── budget_api.json
├── cbr_rates.xml
└── rosstat_report.pdf
```


При классификации признаков учитывается их смысл, а не технический тип хранения. В соответствии с лекцией используются шкалы: номинальная (ID), контентная, категориальная, дихотомическая, порядковая, интервальная, отношений и абсолютная.

In [ ]:
# %pip install openpyxl xlrd beautifulsoup4 pypdf

In [2]:
from pathlib import Path
from io import BytesIO
import json
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [3]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

treasury_zip = data_dir / "treasury.zip"

rosstat_rar = data_dir / "rosstat.rar"
rosstat_dir = data_dir / "rosstat"

budget_api_json = data_dir / "budget_api.json"

# windows может скрывать расширение XML, поэтому проверяются оба варианта имени.
cbr_xml_candidates = [
    data_dir / "cbr_rates.xml",
    data_dir / "cbr_rates"
]

cbr_xml = next(
    (path for path in cbr_xml_candidates if path.exists()),
    cbr_xml_candidates[0]
)

rosstat_report = data_dir / "rosstat_report.pdf"

budget_api_url = "https://budget.gov.ru/epbs/registry/grants/data?pageSize=100&pageNum=1"

print("treasury.zip:", treasury_zip.exists())
print("rosstat.rar:", rosstat_rar.exists())
print("Папка rosstat:", rosstat_dir.exists())
print("budget_api.json:", budget_api_json.exists())
print("cbr_rates.xml:", cbr_xml.exists())
print("rosstat_report.pdf:", rosstat_report.exists())

if rosstat_rar.exists() and not rosstat_dir.exists():
    print("\nРаспакуйте data/rosstat.rar в папку data/rosstat и перезапустите эту ячейку.")

treasury.zip: True
rosstat.rar: True
Папка rosstat: True
budget_api.json: True
cbr_rates.xml: True
rosstat_report.pdf: True


## Вспомогательные функции

In [4]:
def read_excel_bytes(raw, header=None):
    """Читает Excel по фактическому формату, даже если расширение указано неверно."""
    # xlsx -- zip-контейнер и начинается с PK
    if raw[:2] == b"PK":
        return pd.read_excel(BytesIO(raw), engine="openpyxl", header=header)

    # cтарый бинарный XLS
    if raw[:8] == b"\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1":
        return pd.read_excel(BytesIO(raw), engine="xlrd", header=header)

    raise ValueError("Не удалось определить фактический формат Excel-файла.")


def find_excel_in_zip(path):
    """Рекурсивно ищет Excel-файлы во вложенных ZIP."""
    found = []

    if not path.exists():
        print(f"Файл не найден: {path}")
        return found

    def scan(raw, prefix=""):
        if not zipfile.is_zipfile(BytesIO(raw)):
            return

        with zipfile.ZipFile(BytesIO(raw)) as z:
            for member in z.namelist():
                if member.endswith("/"):
                    continue

                data = z.read(member)
                full_name = f"{prefix}/{member}" if prefix else member
                lower = member.lower()

                # excel проверяется раньше, чем zip, так как xlsx сам является ZIP-контейнером
                if lower.endswith((".xls", ".xlsx")):
                    found.append((full_name, data))
                elif zipfile.is_zipfile(BytesIO(data)):
                    scan(data, full_name)

    scan(path.read_bytes(), path.name)
    return found


def read_first_table_from_zip(path):
    """Ищет первую CSV/Excel-таблицу, в том числе во вложенных ZIP."""
    if not path.exists():
        print(f"Файл не найден: {path}")
        return None, None

    def scan(raw, prefix=""):
        if not zipfile.is_zipfile(BytesIO(raw)):
            return None

        with zipfile.ZipFile(BytesIO(raw)) as z:
            for member in z.namelist():
                if member.endswith("/"):
                    continue

                data = z.read(member)
                full_name = f"{prefix}/{member}" if prefix else member
                lower = member.lower()

                if lower.endswith((".xls", ".xlsx")):
                    try:
                        return full_name, read_excel_bytes(data, header=0)
                    except Exception:
                        pass

                if lower.endswith(".csv"):
                    for enc in ("utf-8-sig", "utf-8", "cp1251"):
                        for sep in (";", ",", "\t"):
                            try:
                                df = pd.read_csv(BytesIO(data), encoding=enc, sep=sep)
                                if df.shape[1] > 1:
                                    return full_name, df
                            except Exception:
                                pass

                if zipfile.is_zipfile(BytesIO(data)):
                    result = scan(data, full_name)
                    if result is not None:
                        return result

        return None

    result = scan(path.read_bytes(), path.name)
    if result is None:
        print("Таблица внутри архива не найдена.")
        return None, None
    return result


def inspect_df(df, n=5):
    if df is None:
        return

    print("Размер:", df.shape)
    print("\nТипы данных:")
    display(df.dtypes.to_frame("dtype"))

    print("\nПропуски:")
    display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))

    print("\nПервые строки:")
    display(df.head(n))

## 1. Федеральное казначейство - это основной источник исследования

Источник: консолидированные бюджеты субъектов РФ.  
Роль: основные показатели распределения и исполнения бюджета.  
Формат: ZIP с вложенными архивами и Excel-файлами.

В выгрузке Казначейства Excel-файлы имеют расширение `.xls`, но часть из них фактически записана в формате XLSX. Поэтому формат определяется по содержимому файла, а не только по расширению.

In [5]:
treasury_excel_files = find_excel_in_zip(treasury_zip)

print("Найдено Excel-файлов:", len(treasury_excel_files))

for name, _ in treasury_excel_files[:15]:
    print(name.replace("\\", "/").split("/")[-1])

Найдено Excel-файлов: 128
0503317-ROSKAZNA_1_18_01092026.xls
0503317-ROSKAZNA_1_20_01092026.xls
0503317-ROSKAZNA_1_22_01092026.xls
0503317-ROSKAZNA_1_23_01092026.xls
0503317-ROSKAZNA_1_24_01092026.xls
0503317-ROSKAZNA_1_25_01092026.xls
0503317-ROSKAZNA_1_26_01092026.xls
0503317-ROSKAZNA_1_27_01092026.xls
0503317-ROSKAZNA_1_28_01092026.xls
0503317-ROSKAZNA_1_29_01092026.xls
0503317-ROSKAZNA_1_30_01092026.xls
0503317-ROSKAZNA_1_31_01092026.xls
0503317-ROSKAZNA_2_18_01092026.xls
0503317-ROSKAZNA_2_20_01092026.xls
0503317-ROSKAZNA_2_22_01092026.xls


В именах файлов формы 0503317:
- "1" - доходы;
- "2" - расходы;
- "3" - источники финансирования.

Так как исследование ориентировано прежде всего на расходы, далее рассматриваются файлы группы "2".

In [6]:
expense_files = [
    (name, raw)
    for name, raw in treasury_excel_files
    if "ROSKAZNA_2_" in name.upper()
]

print("Найдено файлов расходов:", len(expense_files))

for name, _ in expense_files[:10]:
    print(name.replace("\\", "/").split("/")[-1])

Найдено файлов расходов: 12
0503317-ROSKAZNA_2_18_01092026.xls
0503317-ROSKAZNA_2_20_01092026.xls
0503317-ROSKAZNA_2_22_01092026.xls
0503317-ROSKAZNA_2_23_01092026.xls
0503317-ROSKAZNA_2_24_01092026.xls
0503317-ROSKAZNA_2_25_01092026.xls
0503317-ROSKAZNA_2_26_01092026.xls
0503317-ROSKAZNA_2_27_01092026.xls
0503317-ROSKAZNA_2_28_01092026.xls
0503317-ROSKAZNA_2_29_01092026.xls


In [7]:
if expense_files:
    treasury_name, treasury_raw = expense_files[0]

    print("Пример файла:", treasury_name.replace("\\", "/").split("/")[-1])

    # header=None нужен, чтобы сначала увидеть исходную форму и определить реальную строку заголовка
    treasury_sample = read_excel_bytes(treasury_raw, header=None)

    print("Размер исходной формы:", treasury_sample.shape)
    display(treasury_sample.head(25))
else:
    treasury_sample = None
    print("Файлы расходов не найдены.")

Пример файла: 0503317-ROSKAZNA_2_18_01092026.xls


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Размер исходной формы: (110, 250)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249
0,Выборка из форм отчетности,за период: 01.09.2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Форма: 0503317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Информация об исполнении консолидированного бюджета субъекта Российской Федерации и бюджета территориального государ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Ед. измерения: руб.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,консолидированный бюджет субъекта Российской Федерации и территориального государственного внебюджетного фонда,200 *** 9600 0000000000 000\nРасходы бюджета - ИТОГО,20000001000000000000000\nОбщегосударственные вопросы,20000001020000000000000\nФункционирование высшего должностного лица субъекта Российской Федерации и муниципального о...,20000001020000000000100\nРасходы на выплаты персоналу в целях обеспечения выполнения функций государственными (муниц...,20000001020000000000110\nРасходы на выплаты персоналу казенных учреждений,"20000001020000000000112\nИные выплаты персоналу учреждений, за исключением фонда оплаты труда",

### Классификация признаков по аналитическим шкалам

Для основного источника используются следующие поля и шкалы:

In [8]:
treasury_scales = pd.DataFrame([
    ["Субъект РФ", "Белгородская область", "Категориальная", "конечное множество регионов без порядка"],
    ["Раздел / подраздел расходов", "Общегосударственные вопросы", "Категориальная", "категории расходов без естественного порядка"],
    ["Код бюджетной классификации", "код КБК", "Номинальная (ID)", "код идентифицирует бюджетную категорию, арифметика над кодом не имеет смысла"],
    ["Сумма бюджетных расходов", "денежная сумма, руб.", "Отношений", "есть естественный ноль, допустима смена единицы измерения"],
    ["Отчетный период", "01.09.2026", "Интервальная", "для дат осмыслены интервалы между моментами времени"]
], columns=["Поле", "Пример", "Шкала", "Обоснование"])

display(treasury_scales)

,Поле,Пример,Шкала,Обоснование
0,Субъект РФ,Белгородская область,Категориальная,конечное множество регионов без порядка
1,Раздел / подраздел расходов,Общегосударственные вопросы,Категориальная,категории расходов без естественного порядка
2,Код бюджетной классификации,код КБК,Номинальная (ID),"код идентифицирует бюджетную категорию, арифметика над кодом не имеет смысла"
3,Сумма бюджетных расходов,"денежная сумма, руб.",Отношений,"есть естественный ноль, допустима смена единицы измерения"
4,Отчетный период,01.09.2026,Интервальная,для дат осмыслены интервалы между моментами времени


## 2. Росстат - факторы влияния

Источник: социально-экономические показатели субъектов РФ.  
Роль: характеристики регионов, которые могут объяснять различия в бюджетных расходах.  
Формат: RAR-архив с Excel-файлами, распакованный в папку "data/rosstat".


In [9]:
rosstat_files = (
    list(rosstat_dir.rglob("*.xlsx"))
    + list(rosstat_dir.rglob("*.xls"))
    + list(rosstat_dir.rglob("*.csv"))
)

print("Найдено табличных файлов Росстата:", len(rosstat_files))

keywords = [
    "Население",
    "Труд",
    "Уровень жизни",
    "Образование",
    "Здравоохранение",
    "Валовой региональный продукт",
    "Инвестиции"
]

selected_rosstat_files = [
    file for file in rosstat_files
    if any(keyword.lower() in file.name.lower() for keyword in keywords)
]

print("\nВыбраны для дальнейшего анализа:")
for file in selected_rosstat_files:
    print(file.name)

Найдено табличных файлов Росстата: 22

Выбраны для дальнейшего анализа:
Раздел 1 - Население.xlsx
Раздел 10 - Инвестиции.xlsx
Раздел 2 - Труд.xlsx
Раздел 3 - Уровень жизни населения.xlsx
Раздел 4 - Образование.xlsx
Раздел 5 - Здравоохранение.xlsx
Раздел 8 - Валовой региональный продукт.xlsx


In [10]:
rosstat_sample = None

population_files = [
    file for file in selected_rosstat_files
    if "Раздел 1 - Население" in file.name
]

if population_files:
    rosstat_file = population_files[0]
elif selected_rosstat_files:
    rosstat_file = selected_rosstat_files[0]
else:
    rosstat_file = None

if rosstat_file is not None:
    print("Пример файла:", rosstat_file.name)

    if rosstat_file.suffix.lower() in (".xlsx", ".xls"):
        rosstat_sample = pd.read_excel(rosstat_file, header=None)

    elif rosstat_file.suffix.lower() == ".csv":
        for enc in ("utf-8-sig", "utf-8", "cp1251"):
            for sep in (";", ",", "\t"):
                try:
                    tmp = pd.read_csv(
                        rosstat_file,
                        encoding=enc,
                        sep=sep,
                        header=None
                    )

                    if tmp.shape[1] > 1:
                        rosstat_sample = tmp
                        break

                except Exception:
                    pass

            if rosstat_sample is not None:
                break

    if rosstat_sample is not None:
        print("Размер исходной таблицы:", rosstat_sample.shape)
        display(rosstat_sample.head(25))
    else:
        print("Не удалось автоматически прочитать выбранный файл.")

else:
    print("Подходящие файлы Росстата не найдены.")

Пример файла: Раздел 1 - Население.xlsx
Размер исходной таблицы: (43, 2)


,0,1
0,NaN,NaN
1,Социально-экономические показатели по субъектам Российской Федерации,NaN
2,1. НАСЕЛЕНИЕ,NaN
3,1.1.,Численность населения
4,1.2.,Среднегодовая численность населения
5,1.3.,Удельный вес городского населения в общей численности населения
6,1.4.,Удельный вес сельского населения в общей численности населения
7,1.5.,Соотношение мужчин и женщин
8,1.6.,Возрастной состав населения
9,1.6.1.,Население моложе трудоспособного возраста


In [11]:
if rosstat_file is not None and rosstat_file.exists():
    xls = pd.ExcelFile(rosstat_file)
    print("Листы файла:")
    print(xls.sheet_names)
else:
    xls = None
    print("Файл Росстата не найден.")

Листы файла:
['Раздел 1', '1.1.', '1.2.', '1.3.', '1.4.', '1.5.', '1.6.1.', '1.6.2.', '1.6.3.', '1.7.1.', '1.7.2.', '1.7.3.', '1.8.', '1.9.', '1.10.', '1.11.', '1.12.', '1.13.', '1.14.', '1.15.', '1.16.1.', '1.16.2.', '1.16.3.', '1.17.', '1.18.', '1.19.', '1.20.', '1.21.1.', '1.21.2.', '1.21.3.', '1.22.1', '1.22.2.', '1.22.3.', '1.23.', '1.24.', '1.25.']


Файл Росстата состоит из отдельных листов по каждому показателю. Для проверки структуры данных рассматривается лист "1.1." - "Численность населения".

In [12]:
population_df = None

if xls is not None and "1.1." in xls.sheet_names:
    population_df = pd.read_excel(
        rosstat_file,
        sheet_name="1.1.",
        header=None
    )

    print("Размер таблицы:", population_df.shape)
    display(population_df.head(30))
else:
    print('Лист "1.1." не найден или файл Росстата недоступен.')

Размер таблицы: (106, 25)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Социально-экономические показатели по субъектам Российской Федерации,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1. НАСЕЛЕНИЕ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.1. ЧИСЛЕННОСТЬ НАСЕЛЕНИЯ1),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(оценка на конец года; тысяч человек),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,2000.0,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,2010,2011,2012,2013,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0
6,Российская Федерация,146303.6,145649.3,144963.6,144333.6,143801.0,143236.6,142862.7,142747.5,142737.2,142833.5,142865.4,143171,143585.9,144025.3,146744.0,147182.3,147580.0,147797.1,147840.7,147959.3,147455.7,146980.1,146447.4,146150.8
7,Центральный федеральный округ,38175.1,38068.5,37946.8,37965.1,38044.0,38109.0,38183.4,38210.1,38263.3,38334.8,38445.7,38640.9,38891.7,39142.4,39383.0,39650.3,39866.9,40078.2,40254.0,40418.4,40341.0,40298.1,40240.3,40198.7
8,Белгородская область,1507.0,1508.1,1511.9,1513.9,1511.7,1511.7,1514.2,1520.1,1526.3,1531.8,1532.4,1536.6,1541.9,1545.5,1549.8,1552.4,1555.5,1553.0,1550.9,1553.0,1545.5,1536.5,1514.5,1500.7
9,Брянская область,1407.9,1391.4,1375.0,1360.2,1344.1,1327.7,1312.7,1303.3,1294.3,1286.5,1275.3,1263.7,1252.2,1240.6,1230.3,1223.1,1218.0,1208.2,1197.1,1189.2,1179.1,1164.6,1152.5,1142.4


### Классификация признаков по аналитическим шкалам

Для табличных данных Росстата классификация строится по фактической структуре листов с показателями.

In [13]:
rosstat_scales = pd.DataFrame([
    ["Субъект РФ", "Брянская область", "Категориальная", "регионы являются категориями без порядка"],
    ["Код показателя", "1.1.", "Номинальная (ID)", "код служит идентификатором показателя"],
    ["Название показателя", "Численность населения", "Категориальная", "название определяет вид показателя"],
    ["Год", "2000, 2001, ...", "Интервальная", "между годами осмыслены равные интервалы"],
    ["Численность населения", "1507.0 тыс. человек", "Отношений", "есть естественный ноль и возможна смена единицы: человек / тыс. человек"],
    ["Доходы, ВРП, инвестиции и другие денежные показатели", "числовое значение", "Отношений", "естественный ноль и допустима смена денежной единицы"]
], columns=["Поле", "Пример", "Шкала", "Обоснование"])

display(rosstat_scales)

,Поле,Пример,Шкала,Обоснование
0,Субъект РФ,Брянская область,Категориальная,регионы являются категориями без порядка
1,Код показателя,1.1.,Номинальная (ID),код служит идентификатором показателя
2,Название показателя,Численность населения,Категориальная,название определяет вид показателя
3,Год,"2000, 2001, ...",Интервальная,между годами осмыслены равные интервалы
4,Численность населения,1507.0 тыс. человек,Отношений,есть естественный ноль и возможна смена единицы: человек / тыс. человек
5,"Доходы, ВРП, инвестиции и другие денежные показатели",числовое значение,Отношений,естественный ноль и допустима смена денежной единицы


Если численность населения будет приведена к целому числу людей и рассматриваться именно как число объектов счета, ее можно интерпретировать как абсолютную шкалу. В исходной таблице показатель задан в тыс. человек, поэтому в работе он относится к шкале отношений.

## 3. "budget.gov.ru" - API с JSON

Источник: реестр соглашений о предоставлении субсидий, бюджетных инвестиций и межбюджетных трансфертов.  
Роль: дополнительная информация о движении бюджетных средств.  
Формат: JSON через API.

In [12]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Connection": "close"
}

payload = None

try:
    response = requests.get(
        budget_api_url,
        headers=headers,
        timeout=(10, 30)
    )

    print("HTTP:", response.status_code)
    response.raise_for_status()

    payload = response.json()
    print("JSON получен через API.")

    with open(budget_api_json, "w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)

except requests.RequestException as error:
    print("API не ответил:")
    print(type(error).__name__, str(error)[:300])

    if budget_api_json.exists():
        print("\nИспользуется ранее сохраненный файл budget_api.json.")

        with open(budget_api_json, "r", encoding="utf-8") as file:
            payload = json.load(file)

    else:
        print(
            "\nЕсли JSON открывается в браузере, сохраните его как "
            "data/budget_api.json и запустите ячейку повторно."
        )

if isinstance(payload, dict):
    print("\nВерхний уровень JSON:", list(payload.keys())[:30])

elif payload is not None:
    print("\nТип JSON:", type(payload).__name__)

HTTP: 200
JSON получен через API.

Верхний уровень JSON: ['offset', 'pageNum', 'pageSize', 'recordCount', 'data', 'order', 'orderDirection', 'limits', 'version', 'pageCount']


In [13]:
if payload is not None:

    if isinstance(payload, dict):
        records = payload.get("data", [])
    elif isinstance(payload, list):
        records = payload
    else:
        records = []

    print("Найдено записей:", len(records))

    api_df = pd.json_normalize(records)

    if not api_df.empty:
        print("Размер таблицы:", api_df.shape)
        display(api_df.head())
        print("\nКолонки:")
        print(api_df.columns.tolist())

    else:
        print("В поле data записи не найдены.")

else:
    records = []
    api_df = pd.DataFrame()
    print("JSON пока не загружен.")

Найдено записей: 100
Размер таблицы: (100, 92)


,id,parentid,childcount,receiver,plans,faip,faipSubject,plansSubject,payments,documents,changes,marks,npa,bo,construct,subjectNpa,infosub,infocost,infoind,controlpoint,addagreement,typereport,mba,reportdocs,soccertificate,informservice,listservice,achievprfrm,info.name,info.code,info.numAgreem,info.dateAgreem,info.regNum,info.numberNpa,info.dateReg,info.nameNpa,info.startDate,info.endDate,info.sum,info.rate,info.currencySum,info.currencyName,info.currencyCode,info.dateUpdate,info.sumMba,info.sumMbamo,info.regnumRgz,info.mfCode,info.mfName,info.npaKind,info.sumSubFzFb,info.outerSystem,info.internaldocnum,info.loaddate,info.ts_mrk,info.ap_mrk,info.t_summ,info.sumtransfers,info.socialnum,info.actdirection,grbs.okopf,grbs.fullName,grbs.shortName,grbs.location,grbs.inn,grbs.kpp,grbs.ogrn,grbs.dateAccount,grbs.kbkInput,grbs.grbsAccount,grbs.codeReestr,grbs.countryCode,grbs.countryName,grbs.regionCode,grbs.regionName,grbs.districtName,grbs.settleName,grbs.postIndex,grbs.locationOktmo,grbs.localCode,grbs.localName,grbs.structType,grbs.streetType,grbs.objectType,grbs.buildingType,grbs.roomType,grbs.tofkcode,grbs.tofkname,grbs.budgtypename,grbs.budgtypecode,grbs.budgetname,grbs.budgetcode
0,117468,,0,"[{'budgetName': '', 'oktmo': '', 'fullName': 'Государственное научное учреждение Московский научно-исследовательский...",[],[],[],[],"[{'num': '131015', 'date': '20140410', 'sumCurrency': '47603.7', 'sumRub': '47603.7', 'rate': '1', 'currencyName': '...","[{'size': '20352', 'name': 'ПП_1_N_895_06.06.14.PDF', 'date': '2014-06-06 01:00:00.0', 'loaddate': '', 'contentType'...","[{'num': '00714957000008', 'numberdoc': '000000000001067', 'text': '008|MGR', 'date': '20140127', 'guid': 'fce5aed0-...",[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],,,007007/У2599/608,2014-01-23 01:00:00.0,00714957000,,,,,,0,1,0,рубль,643,2015-02-14 00:00:00.0,,,,,,,,ASFK,0,,,,,,,,20904,Федеральное агентство научных организаций,ФАНО России,"73,г. Москва,119334,Город Москва,45398000,проспект,Проспект Ленинский,дом № 32А",7736666554,773601001,,2013-10-30 01:00:00.0,007,03951000070,00007,,,73,г. Москва,,,119334,45398000,Город,Москва,проспект,Проспект Ленинский,дом № 32А,,,9500,Межрегиональное операционное управление Федерального казначейства,Федеральный бюджет,01,,
1,121961,,0,"[{'budgetName': '', 'oktmo': '', 'fullName': 'ОАО ""Воронежсельмаш""', 'shortName': '', 'fullNameLat': '', 'okopf': '0...",[],[],[],[],[],"[{'size': '3305376', 'name': 'БО U14_02G25310002_07407081436482810242.zip', 'date': '', 'loaddate': '2020-07-13 16:3...","[{'num': '07414232000008', 'numberdoc': 'U14_02G25310002', 'text': 'MGR', 'date': '-1', 'guid': 'FCE5AED0-5C63-70A8-...",[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],,,02.G25.31.0002,2013-02-12 01:00:00.0,07414232000,,,,,,0,1,0,Российский рубль,RUB,2020-07-13 00:00:00.0,,,,,,,,ASFK,0,2020-07-13 16:36:44.0,,,,,,,0,Министерство образования и науки Российской Федерации,Министерство образования и науки Российской Федерации,"73,Москва,125993,город Москва,45382000,",7710539135,771001001,,2004-04-23 01:00:00.0,074,03951000740,00074,,,73,Москва,,,125993,45382000,город,Москва,,,,,,9500,Межрегиональное операционное управление Федерального казначейства,Федеральный бюджет,01,,
2,121970,,0,"[{'budgetName': '', 'oktmo': '', 'fullName': 'федеральное государственное автономное образовательное учреждение высш...",[],[],[],[],[],"[{'size': '490143', 'name': 'БО I14_14Y26310004_07407081436146622241.zip', 'date': '', 'loaddate': '2020-07-13 16:39...","[{'num': '07414241000008', 'numberdoc': 'I14_14Y26310004', 'text': 'MGR', 'date': '-1', 'guid': 'FCE5AED0-5CE0-70A8-...",[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],,,14.Y26.31.0004,2014-03-18 01:00:00.0,07414241000,,,,,,0,1,0,Российский рубль,RUB,2020-07-13 00:00:00.0,,,,,,,,ASFK,0,2020-07-13 16:36:44.0,,,,,,,0,Министерство образования и науки Российской Федерации,Министерство образования и науки Российской Федерации,"73,Москва,125993,город Москва,45382000,",7710539135,771001001,,2004-04-23 01:


Колонки:
['id', 'parentid', 'childcount', 'receiver', 'plans', 'faip', 'faipSubject', 'plansSubject', 'payments', 'documents', 'changes', 'marks', 'npa', 'bo', 'construct', 'subjectNpa', 'infosub', 'infocost', 'infoind', 'controlpoint', 'addagreement', 'typereport', 'mba', 'reportdocs', 'soccertificate', 'informservice', 'listservice', 'achievprfrm', 'info.name', 'info.code', 'info.numAgreem', 'info.dateAgreem', 'info.regNum', 'info.numberNpa', 'info.dateReg', 'info.nameNpa', 'info.startDate', 'info.endDate', 'info.sum', 'info.rate', 'info.currencySum', 'info.currencyName', 'info.currencyCode', 'info.dateUpdate', 'info.sumMba', 'info.sumMbamo', 'info.regnumRgz', 'info.mfCode', 'info.mfName', 'info.npaKind', 'info.sumSubFzFb', 'info.outerSystem', 'info.internaldocnum', 'info.loaddate', 'info.ts_mrk', 'info.ap_mrk', 'info.t_summ', 'info.sumtransfers', 'info.socialnum', 'info.actdirection', 'grbs.okopf', 'grbs.fullName', 'grbs.shortName', 'grbs.location', 'grbs.inn', 'grbs.kpp', 'grbs.og

После получения JSON отдельно рассматриваются поля, которые могут быть полезны для дальнейшего анализа. Идентификаторы и коды не относятся к числовым аналитическим признакам, даже если записаны числами.

In [14]:
candidate_api_columns = [
    "info.name",
    "info.startDate",
    "info.endDate",
    "info.sum",
    "info.currencySum",
    "info.sumtransfers",
    "grbs.regionName",
    "grbs.districtName",
    "grbs.budgetname"
]

existing_api_columns = [
    column for column in candidate_api_columns
    if column in api_df.columns
]

print("Найдено выбранных полей:", len(existing_api_columns))
print(existing_api_columns)

if existing_api_columns:
    display(api_df[existing_api_columns].head(10))

Найдено выбранных полей: 9
['info.name', 'info.startDate', 'info.endDate', 'info.sum', 'info.currencySum', 'info.sumtransfers', 'grbs.regionName', 'grbs.districtName', 'grbs.budgetname']


,info.name,info.startDate,info.endDate,info.sum,info.currencySum,info.sumtransfers,grbs.regionName,grbs.districtName,grbs.budgetname
0,,,,0,0,,г. Москва,,
1,,,,0,0,,Москва,,
2,,,,0,0,,Москва,,
3,"Соглашения (договора), заключенные ГРБС о предоставлении субсидий из федерального бюджета.",2014-05-30 01:00:00.0,2014-12-31 00:00:00.0,0,0,,Москва,,
4,Нормативно правовой акт.,2014-12-26 00:00:00.0,2999-01-01 00:00:00.0,0,0,,Москва,,
5,Нормативно правовой акт.,2014-09-29 01:00:00.0,2014-09-30 01:00:00.0,0,0,,г. Москва,,
6,Нормативно правовой акт.,,2999-01-01 00:00:00.0,0,0,,Москва,,
7,Нормативно правовой акт.,,2999-01-01 00:00:00.0,0,0,,Москва,,
8,Нормативно правовой акт.,,2999-01-01 00:00:00.0,0,0,,Москва,,
9,Нормативно правовой акт.,,2999-01-01 00:00:00.0,0,0,,Москва,,


In [15]:
if existing_api_columns:
    print("Типы данных:")
    display(api_df[existing_api_columns].dtypes.to_frame("dtype"))

    print("Пропуски:")
    display(
        api_df[existing_api_columns]
        .isna()
        .sum()
        .sort_values(ascending=False)
        .to_frame("missing")
    )

Типы данных:


,dtype
info.name,str
info.startDate,str
info.endDate,str
info.sum,str
info.currencySum,str
info.sumtransfers,str
grbs.regionName,str
grbs.districtName,str
grbs.budgetname,str


Пропуски:


,missing
info.name,0
info.startDate,0
info.endDate,0
info.sum,0
info.currencySum,0
info.sumtransfers,0
grbs.regionName,0
grbs.districtName,0
grbs.budgetname,0


### Классификация признаков по аналитическим шкалам

JSON содержит как обычные поля, так и вложенные объекты и массивы. Поэтому здесь используются одновременно описательная, смысловая и измерительная ветви шкал.

In [16]:
api_scales = pd.DataFrame([
    ["id", "117468", "Номинальная (ID)", "идентификатор записи"],
    ["parentid", "идентификатор родительской записи", "Номинальная (ID)", "используется для связи записей"],
    ["receiver", "вложенный JSON с получателем", "Контентная", "слабоструктурированный объект JSON"],
    ["documents", "массив документов", "Контентная", "слабоструктурированный массив JSON"],
    ["info.name", "Нормативно правовой акт", "Категориальная", "тип / наименование записи"],
    ["grbs.regionName", "Москва", "Категориальная", "регион как категория без порядка"],
    ["grbs.districtName", "федеральный округ", "Категориальная", "округ как категория без порядка"],
    ["grbs.budgetname", "наименование бюджета", "Категориальная", "вид бюджета"],
    ["info.sum", "сумма соглашения", "Отношений", "денежная величина с естественным нулем"],
    ["info.currencySum", "сумма в валюте", "Отношений", "денежная величина с естественным нулем"],
    ["info.sumtransfers", "сумма трансфертов", "Отношений", "денежная величина с естественным нулем"],
    ["childcount", "0, 1, 2, ...", "Абсолютная", "число дочерних объектов с фиксированной единицей счета"],
    ["info.startDate", "дата начала", "Интервальная", "осмыслены интервалы между датами"],
    ["info.endDate", "дата окончания", "Интервальная", "осмыслены интервалы между датами"]
], columns=["Поле", "Пример", "Шкала", "Обоснование"])

display(api_scales)

,Поле,Пример,Шкала,Обоснование
0,id,117468,Номинальная (ID),идентификатор записи
1,parentid,идентификатор родительской записи,Номинальная (ID),используется для связи записей
2,receiver,вложенный JSON с получателем,Контентная,слабоструктурированный объект JSON
3,documents,массив документов,Контентная,слабоструктурированный массив JSON
4,info.name,Нормативно правовой акт,Категориальная,тип / наименование записи
5,grbs.regionName,Москва,Категориальная,регион как категория без порядка
6,grbs.districtName,федеральный округ,Категориальная,округ как категория без порядка
7,grbs.budgetname,наименование бюджета,Категориальная,вид бюджета
8,info.sum,сумма соглашения,Отношений,денежная величина с естественным нулем
9,info.currencySum,сумма в валюте,Отношений,денежная величина с естественным нулем


Поля "info.sum", "info.currencySum" и "info.sumtransfers" в JSON могут приходить как строки. Это технический тип хранения и перед анализом они будут приведены к числовому типу, при этом аналитическая шкала остается шкалой отношений.

## 4. Банк России - XML

Источник: официальные курсы валют Банка России.  
Роль: дополнительный макроэкономический фактор, который может косвенно влиять на стоимость импортных товаров, оборудования и других закупаемых ресурсов.  
Формат: XML.

Файл сохранен в папку "data" как "cbr_rates.xml".


In [16]:
if cbr_xml.exists():
    tree = ET.parse(cbr_xml)
    root = tree.getroot()

    print("Корневой элемент:", root.tag)
    print("Дата курса:", root.attrib.get("Date"))

    rows = []

    for valute in root.findall("Valute"):
        rows.append({
            "id": valute.attrib.get("ID"),
            "num_code": valute.findtext("NumCode"),
            "char_code": valute.findtext("CharCode"),
            "nominal": valute.findtext("Nominal"),
            "name": valute.findtext("Name"),
            "value": valute.findtext("Value"),
            "vunit_rate": valute.findtext("VunitRate")
        })

    cbr_df = pd.DataFrame(rows)

    print("Размер таблицы:", cbr_df.shape)
    display(cbr_df.head(10))

else:
    print("Файл cbr_rates.xml не найден в папке data.")

Корневой элемент: ValCurs
Дата курса: 10.10.2026
Размер таблицы: (54, 7)


,id,num_code,char_code,nominal,name,value,vunit_rate
0,R01010,036,AUD,1,Австралийский доллар,"59,2551","59,2551"
1,R01020A,944,AZN,1,Азербайджанский манат,"49,9440","49,944"
2,R01030,012,DZD,100,Алжирских динаров,"63,1616","0,631616"
3,R01035,826,GBP,1,Фунт стерлингов,"112,2951","112,2951"
4,R01060,051,AMD,100,Армянских драмов,"23,4589","0,234589"
5,R01080,048,BHD,1,Бахрейнский динар,"225,7619","225,7619"
6,R01090B,933,BYN,1,Белорусский рубль,"27,9366","27,9366"
7,R01105,068,BOB,10,Боливиано,"71,6496","7,16496"
8,R01115,986,BRL,1,Бразильский реал,"16,9417","16,9417"
9,R01135,348,HUF,100,Форинтов,"26,1077","0,261077"


После чтения XML числовые значения приводятся к числовому типу, так как в исходном файле десятичные значения записаны через запятую.


In [17]:
if cbr_xml.exists():
    for column in ["nominal", "value", "vunit_rate"]:
        cbr_df[column] = (
            cbr_df[column]
            .astype(str)
            .str.replace(",", ".", regex=False)
        )

        cbr_df[column] = pd.to_numeric(
            cbr_df[column],
            errors="coerce"
        )

    print("Типы данных:")
    display(cbr_df.dtypes.to_frame("dtype"))

    print("Пропуски:")
    display(cbr_df.isna().sum().to_frame("missing"))

Типы данных:


,dtype
id,str
num_code,str
char_code,str
nominal,int64
name,str
value,float64
vunit_rate,float64


Пропуски:


,missing
id,0
num_code,0
char_code,0
nominal,0
name,0
value,0
vunit_rate,0


### Классификация признаков по аналитическим шкалам

Для XML Банка России используются поля элемента `Valute` и дата корневого элемента.

In [15]:
cbr_scales = pd.DataFrame([
    ["ID валюты", "R01010", "Номинальная (ID)", "уникальный идентификатор валюты"],
    ["NumCode", "036", "Номинальная (ID)", "цифровой код валюты, арифметика над ним не имеет смысла"],
    ["CharCode", "AUD", "Номинальная (ID)", "буквенный код валюты"],
    ["Name", "Австралийский доллар", "Категориальная", "наименование валюты как категория"],
    ["Nominal", "1, 10, 100", "Отношений", "количество единиц валюты, есть естественный ноль"],
    ["Value", "официальный курс в рублях", "Отношений", "денежная величина с естественным нулем"],
    ["VunitRate", "курс одной единицы валюты", "Отношений", "денежная величина с естественным нулем"],
    ["Date", "дата курса", "Интервальная", "осмыслены интервалы между датами"]
], columns=["Поле", "Пример", "Шкала", "Обоснование"])

display(cbr_scales)

,Поле,Пример,Шкала,Обоснование
0,ID валюты,R01010,Номинальная (ID),уникальный идентификатор валюты
1,NumCode,036,Номинальная (ID),"цифровой код валюты, арифметика над ним не имеет смысла"
2,CharCode,AUD,Номинальная (ID),буквенный код валюты
3,Name,Австралийский доллар,Категориальная,наименование валюты как категория
4,Nominal,"1, 10, 100",Отношений,"количество единиц валюты, есть естественный ноль"
5,Value,официальный курс в рублях,Отношений,денежная величина с естественным нулем
6,VunitRate,курс одной единицы валюты,Отношений,денежная величина с естественным нулем
7,Date,дата курса,Интервальная,осмыслены интервалы между датами


## 5. Неструктурированные данные - материалы Росстата

Источник: доклад «Социально-экономическое положение России» за январь-август 2026 года.  
Роль: текстовый контекст для интерпретации социально-экономической ситуации и факторов изменения бюджетных показателей.  
Формат: PDF.


In [20]:
if rosstat_report.exists():
    reader = PdfReader(rosstat_report)

    print("Количество страниц:", len(reader.pages))

    pages_to_read = min(20, len(reader.pages))
    pages_text = []

    for page in reader.pages[:pages_to_read]:
        text = page.extract_text()

        if text:
            pages_text.append(text)

    full_report_text = "\n".join(pages_text)

    print("Извлечено страниц:", pages_to_read)
    print("Количество символов:", len(full_report_text))
    print("Количество слов:", len(full_report_text.split()))

    print()
    print(full_report_text[:100000])

else:
    full_report_text = ""
    print("Файл rosstat_report.pdf не найден в папке data.")

Количество страниц: 278
Извлечено страниц: 20
Количество символов: 42390
Количество слов: 4355

Социально-экономическое 
положение России
Москва
январь-август 2026 года
8
 
МИНЭКОНОМРАЗВИТИЯ РОССИИ 
 
ФЕДЕРАЛЬНАЯ СЛУЖБА ГОСУДАРСТВЕННОЙ СТАТИСТИКИ 
(РОССТАТ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
СОЦИАЛЬНО-ЭКОНОМИЧЕСКОЕ 
ПОЛОЖЕНИЕ РОССИИ 
 
январь-август 2026 года 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
МОСКВА 

 
 3 
СОДЕРЖАНИЕ 
 
 
                                 Стр.  
  
I. ОСНОВНЫЕ ЭКОНОМИЧЕСКИЕ И СОЦИАЛЬНЫЕ ПОКАЗАТЕЛИ .........................................................................................  6 
II. ЭКОНОМИЧЕСКАЯ СИТУАЦИЯ В РОССИЙСКОЙ ФЕДЕРАЦИИ ............................................................................................  7 
  1. Производство товаров и услуг .................................................................................................................................................................  7 
        1.1. Валовой внутренний продукт ............

Для проверки структуры неструктурированного источника извлекается текст из первых 20 страниц доклада. На следующем этапе этот текст можно использовать для аннотирования и векторизации.

Для исследования выбран наиболее актуальный доклад Росстата за январь-август 2026 года. Доклад позволяет дополнить числовые показатели контекстом о текущей социально-экономической ситуации.


### Классификация признаков по аналитическим шкалам

Для неструктурированного источника основной объект анализа - сам текст. Дополнительные поля появляются после разметки документа.

In [14]:
report_scales = pd.DataFrame([
    ["Текст доклада", "извлеченный текст PDF", "Контентная", "свободный неструктурированный текст"],
    ["Название раздела после аннотирования", "Производство товаров и услуг", "Категориальная", "тематическая категория без порядка"],
    ["Период доклада", "январь-август 2026", "Интервальная", "временной интервал документа"]
], columns=["Поле", "Пример", "Шкала", "Обоснование"])

display(report_scales)

,Поле,Пример,Шкала,Обоснование
0,Текст доклада,извлеченный текст PDF,Контентная,свободный неструктурированный текст
1,Название раздела после аннотирования,Производство товаров и услуг,Категориальная,тематическая категория без порядка
2,Период доклада,январь-август 2026,Интервальная,временной интервал документа


Дихотомическая и порядковая шкалы среди выбранных исходных полей сейчас не используются. Они могут появиться позже у производных признаков, например "аномалия / не аномалия" или у ранга приоритета.

## Итоговый набор источников

In [21]:
sources = pd.DataFrame([
    {
        "Источник": "Федеральное казначейство",
        "Формат": "табличный, ZIP/Excel",
        "Роль": "основные бюджетные показатели",
        "Связь": "субъект РФ + период + направление расходов",
    },
    {
        "Источник": "Росстат",
        "Формат": "табличный, RAR → распакованные файлы",
        "Роль": "социально-экономические факторы",
        "Связь": "субъект РФ + период",
    },
    {
        "Источник": "budget.gov.ru",
        "Формат": "JSON API",
        "Роль": "соглашения, субсидии и трансферты",
        "Связь": "регион / получатель + период",
    },
    {
        "Источник": "Банк России",
        "Формат": "XML",
        "Роль": "валютные курсы как дополнительный макроэкономический фактор",
        "Связь": "валюта + дата",
    },
    {
        "Источник": "Росстат: доклады",
        "Формат": "PDF, неструктурированный текст",
        "Роль": "контекст социально-экономической ситуации",
        "Связь": "регион / период",
    },
])

sources

,Источник,Формат,Роль,Связь
0,Федеральное казначейство,"табличный, ZIP/Excel",основные бюджетные показатели,субъект РФ + период + направление расходов
1,Росстат,"табличный, RAR → распакованные файлы",социально-экономические факторы,субъект РФ + период
2,budget.gov.ru,JSON API,"соглашения, субсидии и трансферты",регион / получатель + период
3,Банк России,XML,валютные курсы как дополнительный макроэкономический фактор,валюта + дата
4,Росстат: доклады,"PDF, неструктурированный текст",контекст социально-экономической ситуации,регион / период


## Предварительный вывод

Выбранные источники дополняют друг друга и позволяют рассматривать бюджетные потоки не только как набор отдельных финансовых показателей, но и в более широком социально-экономическом контексте. Данные Федерального казначейства являются основным источником исследования, поскольку они непосредственно отражают структуру и объемы бюджетных расходов субъектов РФ. Показатели Росстата в свою очередь позволяют сопоставить бюджетные характеристики с численностью населения, доходами, ВРП, инвестициями и другими особенностями регионов. API budget.gov.ru дополняет исследование более детальной информацией о бюджетном финансировании, получателях средств и связанных с ними показателях. Данные Банка России используются как дополнительный макроэкономический фактор, который может учитываться при интерпретации отдельных изменений расходов. Неструктурированный доклад Росстата дает содержательный контекст и помогает связать выявленные количественные изменения с социально-экономическими процессами. Совместное использование этих источников позволит проверить аналитическую гипотезу о наличии устойчивых структурных и временных закономерностей бюджетных расходов, а также оценить, связаны ли различия между субъектами РФ с их социально-экономическими характеристиками.

Основной объект анализа - бюджетные расходы субъектов РФ.